In [1]:
# Dependencies: pandas numpy scikit-learn matplotlib seaborn joblib


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Shared grouped splits + the agreed 30 features (see codd_common.py).
# Never regenerate the split here: all models must use the same files.
from codd_common import load_grouped_splits, assert_feature_set

data = load_grouped_splits()
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
X_train, X_val, X_test = data.X_train, data.X_val, data.X_test
y_train, y_val, y_test = data.y_train, data.y_val, data.y_test
feature_columns = data.feature_columns
numeric_features = data.numeric_features
categorical_features = data.categorical_features

assert_feature_set(X_train.columns)
assert_feature_set(X_val.columns)
assert_feature_set(X_test.columns)
print("Features used:", feature_columns)

# MoE: the gate predicts Material_Type from the remaining 29 features, so the
# preprocessor is built without the gate column. MixtureOfExperts receives the
# full 30-feature frame and drops the gate column internally.
GATE_TARGET = "Material_Type"
RANDOM_STATE = 42
moe_numeric = [c for c in numeric_features if c != GATE_TARGET]
moe_categorical = [c for c in categorical_features if c != GATE_TARGET]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), moe_numeric),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), moe_categorical)
])

print("Condition classes:", sorted(y_train.unique()))
print("Material types:", sorted(X_train[GATE_TARGET].dropna().unique()))


Grouped splits: train (11465, 30) val (2410, 30) test (2644, 30)
Image_ID overlap (must all be 0): {'train-val': 0, 'train-test': 0, 'val-test': 0}
Features: 30 (24 numeric, 6 categorical)
Features used: ['Material_Type', 'Material_Subtype', 'Source', 'Exposure_Type', 'Crack_Level', 'Surface_Damage', 'Breakage_Level', 'Discoloration_Level', 'Deformation_Level', 'Missing_Portion', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']
Condition classes: ['Excellent', 'Fair', 'Good', 'Poor']
Material types: ['brick', 'concrete', 'foam', 'general_w', 'gypsum_board', 'pipes', 'plastic', 'stone', 'tile', 'wood']


In [3]:
# MixtureOfExperts is defined in codd_models.py so saved models can be loaded anywhere.
from codd_models import MixtureOfExperts

# Train using training data only
moe_model = MixtureOfExperts(preprocessor, gate_col=GATE_TARGET, min_samples=30, random_state=RANDOM_STATE)
moe_model.fit(X_train, y_train)

print("Trained experts:", list(moe_model.experts_.keys()))


Trained experts: ['brick', 'concrete', 'tile', 'foam', 'general_w', 'pipes', 'plastic', 'gypsum_board', 'wood', 'stone']


In [4]:
val_pred = moe_model.predict(X_val)

val_accuracy = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)

print("Validation Accuracy:", round(val_accuracy, 4))
print("Validation Macro F1:", round(val_f1, 4))
print(classification_report(y_val, val_pred, zero_division=0))

Validation Accuracy: 0.9382
Validation Macro F1: 0.866
              precision    recall  f1-score   support

   Excellent       0.92      0.52      0.67        21
        Fair       0.98      0.93      0.96      1596
        Good       0.85      0.96      0.90       568
        Poor       0.93      0.96      0.94       225

    accuracy                           0.94      2410
   macro avg       0.92      0.84      0.87      2410
weighted avg       0.94      0.94      0.94      2410



In [5]:
X_train_full = pd.concat([X_train, X_val], ignore_index=True)
y_train_full = pd.concat([y_train, y_val], ignore_index=True)

# Final model: refit on train + validation (hyperparameters are fixed)
final_moe = MixtureOfExperts(preprocessor, gate_col=GATE_TARGET, min_samples=30, random_state=RANDOM_STATE)
final_moe.fit(X_train_full, y_train_full)

test_pred = final_moe.predict(X_test)

test_accuracy = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred, average="macro", zero_division=0)

print("Test Accuracy:", round(test_accuracy, 4))
print("Test Macro F1:", round(test_f1, 4))
print(classification_report(y_test, test_pred, zero_division=0))


Test Accuracy: 0.9508
Test Macro F1: 0.9126
              precision    recall  f1-score   support

   Excellent       0.96      0.72      0.82        32
        Fair       0.98      0.95      0.97      1788
        Good       0.87      0.96      0.91       586
        Poor       0.96      0.94      0.95       238

    accuracy                           0.95      2644
   macro avg       0.94      0.89      0.91      2644
weighted avg       0.95      0.95      0.95      2644



In [6]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

os.makedirs("model_results", exist_ok=True)
os.makedirs("saved_models", exist_ok=True)

moe_metrics = pd.DataFrame([
    {
        "Model": "Mixture of Experts",
        "Split": "Validation",
        "Accuracy": val_accuracy,
        "Macro F1": val_f1
    },
    {
        "Model": "Mixture of Experts",
        "Split": "Test",
        "Accuracy": test_accuracy,
        "Macro F1": test_f1
    }
])

moe_metrics.to_csv(
    "model_results/mixture_of_experts_metrics.csv",
    index=False
)

joblib.dump(final_moe, "saved_models/codd_moe_pipeline.joblib")

cm = confusion_matrix(
    y_test,
    test_pred,
    labels=final_moe.classes_
)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Mixture of Experts - Test Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(moe_metrics)
print("Saved metrics and model.")


                Model       Split  Accuracy  Macro F1
0  Mixture of Experts  Validation  0.938174  0.865983
1  Mixture of Experts        Test  0.950832  0.912613
Saved metrics and model.


C:\Users\Ryan John\AppData\Local\Temp\ipykernel_8140\4200994223.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
